In [1]:
import os, json, time, statistics
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
Path('results').mkdir(exist_ok=True)
Path('results/spark-events').mkdir(exist_ok=True)


In [2]:
import boto3
endpoint = os.getenv('S3_ENDPOINT', 'http://localhost:9000')
access = os.getenv('AWS_ACCESS_KEY_ID', 'hw23student')
secret = os.getenv('AWS_SECRET_ACCESS_KEY', 'hw23student-secret')
client = boto3.client('s3', endpoint_url=endpoint, aws_access_key_id=access, aws_secret_access_key=secret)
if 'hw22' not in [b['Name'] for b in client.list_buckets()['Buckets']]:
    client.create_bucket(Bucket='hw22')
jars = '/opt/airflow/s3-jars/'
s = (SparkSession.builder.master('local[2]').appName('hw22-join')
     .config('spark.driver.memory', '1g').config('spark.executor.memory', '1g')
     .config('spark.sql.session.timeZone', 'UTC').config('spark.sql.shuffle.partitions', '16')
     .config('spark.ui.showConsoleProgress', 'false')
     .config('spark.jars', jars + 'hadoop-aws-3.3.4.jar,' + jars + 'aws-java-sdk-bundle-1.12.262.jar')
     .config('spark.hadoop.fs.s3a.endpoint', endpoint)
     .config('spark.hadoop.fs.s3a.path.style.access', 'true')
     .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
     .config('spark.hadoop.fs.s3a.access.key', access).config('spark.hadoop.fs.s3a.secret.key', secret)
     .config('spark.eventLog.enabled', 'true').config('spark.eventLog.compress', 'false')
     .config('spark.eventLog.dir', Path('results/spark-events').resolve().as_uri()).getOrCreate())
s.sparkContext.setLogLevel('ERROR')
app_id = s.sparkContext.applicationId
scd = s.read.parquet('data/task1/scd2')
c = s.read.schema('campaign_id long,email_domain string,send_at timestamp,campaign_name string').option('header', True).csv('data/task1/campaigns.csv')
assert scd.count() == 1080897
assert c.count() == c.select('campaign_id').distinct().count() == 25
print('Spark', s.version, 'local[2], driver 1 GiB, executor 1 GiB, shuffle partitions 16', 'CPU', os.cpu_count())


26/11/13 19:53:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.4 local[2], driver 1 GiB, executor 1 GiB, shuffle partitions 16 CPU 12


In [3]:
def measurements(app_id):
    stages, executions, plans, tasks = {}, {}, {}, {}
    for path in Path('results/spark-events').glob(app_id + '*'):
        for line in path.open():
            e = json.loads(line)
            kind = e['Event']
            if kind == 'SparkListenerJobStart':
                props = e.get('Properties') or {}
                group = props.get('spark.jobGroup.id')
                if group:
                    stages.setdefault(group, set()).update(e['Stage IDs'])
                    if 'spark.sql.execution.id' in props:
                        executions.setdefault(group, set()).add(int(props['spark.sql.execution.id']))
            elif kind.endswith(('SparkListenerSQLExecutionStart', 'SparkListenerSQLAdaptiveExecutionUpdate')):
                plans[e['executionId']] = e.get('physicalPlanDescription', '')
            elif kind == 'SparkListenerTaskEnd' and e.get('Task End Reason', {}).get('Reason') == 'Success':
                m = e.get('Task Metrics', {})
                read = m.get('Shuffle Read Metrics', {})
                info = e['Task Info']
                tasks.setdefault(e['Stage ID'], []).append({
                    'shuffle_read': read.get('Remote Bytes Read', 0) + read.get('Local Bytes Read', 0),
                    'shuffle_write': m.get('Shuffle Write Metrics', {}).get('Shuffle Bytes Written', 0),
                    'memory_spill': m.get('Memory Bytes Spilled', 0),
                    'disk_spill': m.get('Disk Bytes Spilled', 0),
                    'task_ms': info['Finish Time'] - info['Launch Time']})
    rows = []
    for group, ids in stages.items():
        ts = [t for sid in ids for t in tasks.get(sid, [])]
        row = {'run': group, **{k: sum(t[k] for t in ts) for k in ['shuffle_read', 'shuffle_write', 'memory_spill', 'disk_spill']}}
        durations = [t['task_ms'] for t in ts]
        row.update(task_median_ms=statistics.median(durations) if durations else 0, task_max_ms=max(durations, default=0))
        rows.append(row)
        if group.endswith('_5') or group in ('baseline', 'broadcast', 'salt'):
            print(group)
            for eid in sorted(executions.get(group, [])):
                print(plans.get(eid, ''))
    import pandas as pd
    display(pd.DataFrame(rows))
    return rows

def same(a, b):
    return not a.exceptAll(b.select(a.columns)).limit(1).count() and not b.select(a.columns).exceptAll(a).limit(1).count()


In [4]:
def audience(strategy):
    u = scd.withColumn('domain', F.lower(F.substring_index('user_email', '@', -1))).alias('u')
    camp = c.alias('c')
    if strategy == 'merge':
        u, camp = u.hint('MERGE'), camp.hint('MERGE')
    elif strategy == 'broadcast':
        camp = F.broadcast(camp)
    elif strategy == 'shuffle_hash':
        camp = camp.hint('SHUFFLE_HASH')
    condition = ((F.col('u.domain') == F.lower(F.col('c.email_domain')))
                 & (F.col('u.valid_from') <= F.col('c.send_at'))
                 & (F.col('u.valid_to').isNull() | (F.col('c.send_at') < F.col('u.valid_to'))))
    return u.join(camp, condition, 'inner').select('campaign_id', 'campaign_name', 'send_at', 'email_domain', 'user_id', 'user_fio', 'user_email', 'user_login')
keys = ['spark.sql.adaptive.enabled', 'spark.sql.autoBroadcastJoinThreshold', 'spark.sql.shuffle.partitions']
original = {k: s.conf.get(k) for k in keys}
print('Исходные настройки:', original)
strategies = ['auto', 'merge', 'broadcast', 'shuffle_hash']
runs, outputs = [], {}
for turn in range(6):
    order = strategies[turn % 4:] + strategies[:turn % 4]
    for name in order:
        for k, value in original.items():
            s.conf.set(k, value)
        if name != 'auto':
            for k, value in zip(keys, ['false', '-1', '16']):
                s.conf.set(k, value)
        query = audience(name)
        group = f'{name}_{turn}'
        path = 's3a://hw22/benchmark/' + group
        s.sparkContext.setJobGroup(group, group)
        start = time.perf_counter()
        query.write.mode('overwrite').parquet(path)
        seconds = time.perf_counter() - start
        s.sparkContext.setLocalProperty('spark.jobGroup.id', None)
        actual = s.read.parquet(path)
        runs.append({'strategy': name, 'round': turn, 'seconds': seconds, 'rows': actual.count(), 'files': len(actual.inputFiles())})
        outputs[name] = actual
for name, actual in outputs.items():
    assert same(outputs['auto'], actual)
assert not outputs['auto'].groupBy('campaign_id', 'user_id').count().filter('count > 1').count()
for k, value in original.items():
    s.conf.set(k, value)
import pandas as pd
times = pd.DataFrame(runs)
display(times)
summary = times[times['round'] > 0].groupby('strategy').seconds.agg(['median', 'min', 'max'])
display(summary)
print('Полное равенство с учётом повторов: True; пары campaign_id/user_id уникальны.')
s.stop()


Исходные настройки: {'spark.sql.adaptive.enabled': 'true', 'spark.sql.autoBroadcastJoinThreshold': '10485760b', 'spark.sql.shuffle.partitions': '16'}


,strategy,round,seconds,rows,files
0,auto,0,2.117878,500233,2
1,merge,0,1.993229,500233,5
2,broadcast,0,0.819524,500233,2
3,shuffle_hash,0,1.598555,500233,5
4,merge,1,1.353874,500233,5
5,broadcast,1,0.710324,500233,2
6,shuffle_hash,1,1.309826,500233,5
7,auto,1,0.708150,500233,2
8,broadcast,2,0.704533,500233,2
9,shuffle_hash,2,1.320226,500233,5


,median,min,max
strategy,,,
auto,0.715522,0.685861,0.721293
broadcast,0.710324,0.687249,0.780505
merge,1.344943,1.289426,1.406283
shuffle_hash,1.318881,1.302747,1.474629


Полное равенство с учётом повторов: True; пары campaign_id/user_id уникальны.


In [5]:
metrics = measurements(app_id)
best = summary['median'].idxmin()
print(f'На этом запуске минимальная медиана у {best}: {summary.loc[best, "median"]:.3f} с. Разброс каждого варианта указан в min/max.')
print('Справочник содержит 25 строк: broadcast исключает shuffle большой витрины. Время также включает чтение и запись S3, поэтому уменьшение shuffle не гарантирует меньшую длительность.')
print('Пять доменов дают не более пяти непустых разделов по ключу при shuffle; горячий домен ограничивает параллелизм. При росте справочника, изменении распределения доменов или ресурсов выбор может измениться.')

print("Auto выбрал BroadcastHashJoin, явный broadcast сохранил оператор. Диапазоны времени auto и broadcast пересекаются: устойчивого ускорения относительно auto в этих замерах нет.")


merge_5
== Physical Plan ==
Execute InsertIntoHadoopFsRelationCommand (14)
+- WriteFiles (13)
   +- * Project (12)
      +- * SortMergeJoin Inner (11)
         :- * Sort (6)
         :  +- Exchange (5)
         :     +- * Project (4)
         :        +- * Filter (3)
         :           +- * ColumnarToRow (2)
         :              +- Scan parquet  (1)
         +- * Sort (10)
            +- Exchange (9)
               +- * Filter (8)
                  +- Scan csv  (7)


(1) Scan parquet 
Output [6]: [user_id#0L, user_fio#1, user_email#2, user_login#3, valid_from#4, valid_to#5]
Batched: true
Location: InMemoryFileIndex [file:/work/data/task1/scd2]
PushedFilters: [IsNotNull(valid_from)]
ReadSchema: struct<user_id:bigint,user_fio:string,user_email:string,user_login:string,valid_from:timestamp,valid_to:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [user_id#0L, user_fio#1, user_email#2, user_login#3, valid_from#4, valid_to#5]

(3) Filter [codegen id : 1]
Input [6]: [user_id#0L

,run,shuffle_read,shuffle_write,memory_spill,disk_spill,task_median_ms,task_max_ms
0,auto_0,0,0,0,0,1304,1304
1,merge_0,50888173,50888173,0,0,15,518
2,broadcast_0,0,0,0,0,634,636
3,shuffle_hash_0,50888173,50888173,0,0,10,483
4,merge_1,50888173,50888173,0,0,11,388
5,broadcast_1,0,0,0,0,539,541
6,shuffle_hash_1,50888173,50888173,0,0,10,360
7,auto_1,0,0,0,0,536,539
8,broadcast_2,0,0,0,0,546,548
9,shuffle_hash_2,50888173,50888173,0,0,9,376


На этом запуске минимальная медиана у broadcast: 0.710 с. Разброс каждого варианта указан в min/max.
Справочник содержит 25 строк: broadcast исключает shuffle большой витрины. Время также включает чтение и запись S3, поэтому уменьшение shuffle не гарантирует меньшую длительность.
Пять доменов дают не более пяти непустых разделов по ключу при shuffle; горячий домен ограничивает параллелизм. При росте справочника, изменении распределения доменов или ресурсов выбор может измениться.
Auto выбрал BroadcastHashJoin, явный broadcast сохранил оператор. Диапазоны времени auto и broadcast пересекаются: устойчивого ускорения относительно auto в этих замерах нет.
